<a href="https://colab.research.google.com/github/CraigUni27/Python-Practice/blob/main/HR1_Overlay_Charts_Colab_UPDATED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HydraRacker HR1 — synchronized 5-signal overlay

This notebook uses the **single merged HR1 workbook** containing `Short Description` and plots the five selected signals on **vertically aligned subplots sharing exactly the same time axis**.

Selected signals:
- `HR1-181-A00-st` — Actual Load
- `HR1-352-A00-st` — Lower Trolley Actual Position
- `HR1-515-A00-st` — Mode in text
- `HR1-435-D00-st` — Main Arm Gripper Open
- `HR1-434-D00-st` — Main Arm Gripper Close

Each signal keeps its **own y-axis**, while all five use the same x-axis. Hover timestamps show milliseconds and an optional **500 ms analysis grid** can be switched on after zooming in.

In [1]:
!pip -q install plotly openpyxl xlsxwriter

import io
import re
import html
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from google.colab import files
from IPython.display import display

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 5.8 MB/s eta 0:00:00


## 1. Upload the merged HR1 workbook

In [17]:
from google.colab import files
import pandas as pd

uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_excel(filename)
print(f"Loaded: {filename}")
display(df.head())

Saving tags_2026-09-30_HR1_st.xlsx to tags_2026-09-30_HR1_st (1).xlsx
Loaded: tags_2026-09-30_HR1_st (1).xlsx


,topic,description,Short Description,time,value,Converted Time
0,forth-mqtt/ns2_Objects_NOV_Signals_HR1-203-A00-st,Actual Position,Main Arm Actual Position,1790779979972,0.995279,2026-09-30 14:52:59.972
1,forth-mqtt/ns2_Objects_NOV_Signals_HR1-181-A00-st,Actual Load,Actual Load,1790779979972,-0.219553,2026-09-30 14:52:59.972
2,forth-mqtt/ns2_Objects_NOV_Signals_HR1-352-A00-st,Actual Position,Lower Trolley Actual Position,1790779979675,4.015215,2026-09-30 14:52:59.675
3,forth-mqtt/ns2_Objects_NOV_Signals_HR1-333-A00-st,Actual Position,Upper Trolley Actual Position,1790779979675,4.015093,2026-09-30 14:52:59.675
4,forth-mqtt/ns2_Objects_NOV_Signals_HR1-522-A00-st,Actual Grip Pressure,Main Arm Roller Close Pressure Actual Value,1790779979221,45.601852,2026-09-30 14:52:59.221


## 2. Read and clean the merged HR1 data

In [18]:
raw = pd.read_excel(filename)

required = {"topic", "time", "value"}
missing = required - set(raw.columns)
if missing:
    raise ValueError(f"Workbook is missing required columns: {sorted(missing)}")

# The merged workbook should already contain Short Description.
if "Short Description" not in raw.columns:
    raw["Short Description"] = raw.get("description", "")

def extract_tag(topic):
    s = str(topic)
    if "_Signals_" in s:
        return s.split("_Signals_", 1)[1]
    return re.split(r"[/_]", s)[-1]

raw["Tag"] = raw["topic"].map(extract_tag).astype(str).str.strip()

# Source time = epoch milliseconds in UTC. Display in Europe/London.
raw["Time BST"] = (
    pd.to_datetime(raw["time"], unit="ms", utc=True, errors="coerce")
      .dt.tz_convert("Europe/London")
)

def value_to_number(v):
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    if s.lower() == "true":
        return 1.0
    if s.lower() == "false":
        return 0.0
    try:
        return float(s)
    except Exception:
        return np.nan

raw["Value_numeric"] = raw["value"].map(value_to_number)

data = (
    raw.loc[
        raw["Time BST"].notna(),
        ["Tag", "Short Description", "Time BST", "value", "Value_numeric"]
    ]
    .sort_values("Time BST")
    .reset_index(drop=True)
)

# One description lookup, taken directly from the merged workbook.
desc_lookup = (
    data.dropna(subset=["Short Description"])
        .drop_duplicates("Tag")
        .set_index("Tag")["Short Description"]
        .astype(str)
        .to_dict()
)

def canonical_description(tag):
    return desc_lookup.get(tag, tag)

print(f"Rows loaded: {len(data):,}")
print(f"Unique tags available: {data['Tag'].nunique():,}")
print(f"Time range: {data['Time BST'].min()}  →  {data['Time BST'].max()}")

Rows loaded: 18,308
Unique tags available: 68
Time range: 2026-09-30 15:00:00.060000+01:00  →  2026-09-30 15:52:59.972000+01:00


The merged workbook already contains the cross-referenced `Short Description`, so no second tag-list upload or matching step is required.

## 4. Select the five synchronized signals

In [29]:
SELECTED_TAGS = [
    "HR1-181-A00-st",  # Actual Load
    "HR1-431-D00-st",  # Gripper Head Loaded With Pipe
    "HR1-435-D00-st",  # Main Arm Gripper Open
    "HR1-434-D00-st",  # Main Arm Gripper Close
    "HR1-352-A00-st",  # Lower Trolley Actual Position
]

# Check that all five tags actually exist in the uploaded workbook.
present = set(data["Tag"].dropna().unique())

missing_selected = [
    tag for tag in SELECTED_TAGS
    if tag not in present
]

if missing_selected:
    raise ValueError(
        f"Selected tags not found in workbook: {missing_selected}"
    )

# Preserve the order above for the five stacked plots.
tags = SELECTED_TAGS

print("Selected tags:")
for tag in tags:
    print(f"  {tag} — {canonical_description(tag)}")

Selected tags:
  HR1-181-A00-st — Actual  Load
  HR1-431-D00-st — Gripper Head Loaded With Pipe
  HR1-435-D00-st — Main Arm Gripper Open
  HR1-434-D00-st — Main Arm Gripper Close
  HR1-352-A00-st — Lower Trolley Actual Position


## 5. Create the synchronized overlay

The five traces are **stacked vertically but locked to the same x-axis**. This is preferable to five overlaid scales in one plot because each signal retains a readable y-axis while changes remain perfectly aligned in time.

- Hover shows the raw timestamp to **1 ms**.
- Major time labels are readable at normal zoom.
- Use the **500 ms grid ON** button after zooming into a sequence to show half-second minor grid lines.
- Drag horizontally to zoom; double-click to reset.

In [30]:
from plotly.subplots import make_subplots

# Human-readable y-axis labels in the same order as SELECTED_TAGS.
Y_LABELS = {
    "HR1-181-A00-st": "Actual Load",
    "HR1-352-A00-st": "Lower Trolley Position (m)",
    "HR1-431-A00-st": "Gripper Head Loaded with Pipe",
    "HR1-435-D00-st": "Gripper Open",
    "HR1-434-D00-st": "Gripper Close",
}

fig = make_subplots(
    rows=len(tags),
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.025,
    subplot_titles=[f"{t} — {canonical_description(t)}" for t in tags],
)

summary_rows = []

for row, tag in enumerate(tags, start=1):
    d = data.loc[data["Tag"] == tag].sort_values("Time BST").copy()
    desc = canonical_description(tag)

    # The selected five signals are numeric/boolean in this dataset.
    # Keep raw values; do NOT resample or round them because event timing matters.
    d["PlotValue"] = d["Value_numeric"]

    unique_values = set(np.round(d["PlotValue"].dropna().unique(), 10).tolist())
    is_binary = len(unique_values) <= 2 and unique_values.issubset({0.0, 1.0})


    # State-type signals are drawn as steps; analogue signals as normal lines.
    line_shape = "hv" if is_binary else "linear"

    fig.add_trace(
        go.Scatter(
            x=d["Time BST"],
            y=d["PlotValue"],
            mode="lines",
            line_shape=line_shape,
            name=f"{tag} — {desc}",
            connectgaps=False,
            hovertemplate=(
                f"<b>{tag}</b><br>"
                "%{x|%H:%M:%S.%L}<br>"
                "Value: %{y:.6g}<extra></extra>"
            ),
        ),
        row=row,
        col=1,
    )

    fig.update_yaxes(
        title_text=Y_LABELS.get(tag, desc),
        showgrid=True,
        zeroline=False,
        row=row,
        col=1,
    )

    if is_binary:
        fig.update_yaxes(
            tickmode="array",
            tickvals=[0, 1],
            ticktext=["0 / False", "1 / True"],
            range=[-0.15, 1.15],
            row=row,
            col=1,
        )


    valid = d["PlotValue"].dropna()
    summary_rows.append({
        "Tag": tag,
        "Description": desc,
        "Samples": len(d),
        "First time BST": d["Time BST"].min(),
        "Last time BST": d["Time BST"].max(),
        "Min": valid.min() if len(valid) else np.nan,
        "Max": valid.max() if len(valid) else np.nan,
    })

# All subplots share the exact same x range and scale.
fig.update_xaxes(
    matches="x",
    showgrid=True,
    tickformat="%H:%M:%S.%L",
    showspikes=True,
    spikemode="across",
    spikesnap="cursor",
    spikethickness=1,
)

# Show x-axis labels only on the bottom subplot.
for r in range(1, len(tags)):
    fig.update_xaxes(showticklabels=False, row=r, col=1)

fig.update_xaxes(
    title_text="Time BST — hover for exact milliseconds",
    showticklabels=True,
    row=len(tags),
    col=1,
)

# Buttons: the 500 ms grid is best used after zooming into a short interval.
# Major labels every 5 seconds, minor dotted grid every 500 ms.
grid_on = {}
grid_off = {}
for axis_num in range(1, len(tags) + 1):
    axis_name = "xaxis" if axis_num == 1 else f"xaxis{axis_num}"
    grid_on[f"{axis_name}.dtick"] = 5000
    grid_on[f"{axis_name}.minor.dtick"] = 500
    grid_on[f"{axis_name}.minor.showgrid"] = True
    grid_on[f"{axis_name}.minor.griddash"] = "dot"

    grid_off[f"{axis_name}.dtick"] = None
    grid_off[f"{axis_name}.minor.showgrid"] = False

fig.update_layout(
    title=(
        "HR1 synchronized sequence signals — shared time axis<br>"
        "<sup>Zoom into an event, then switch on the 500 ms grid for half-second alignment</sup>"
    ),
    height=1050,
    width=1250,
    template="plotly_white",
    hovermode="x unified",
    spikedistance=-1,
    showlegend=False,
    margin=dict(l=110, r=40, t=120, b=70),
    updatemenus=[
        dict(
            type="buttons",
            direction="right",
            x=1,
            xanchor="right",
            y=1.08,
            yanchor="bottom",
            buttons=[
                dict(label="500 ms grid ON", method="relayout", args=[grid_on]),
                dict(label="500 ms grid OFF", method="relayout", args=[grid_off]),
            ],
        )
    ],
)

# If supported by the installed Plotly version, extend hover across matched subplots.
try:
    fig.update_layout(hoversubplots="axis")
except Exception:
    pass

fig.show(config={"scrollZoom": True, "displaylogo": False})

# Keep these names for the export cells below.
overlay_fig = fig
figures = [("HR1_Selected_Overlay", "Five synchronized HR1 signals", overlay_fig)]
summary = pd.DataFrame(summary_rows)

display(summary)

,Tag,Description,Samples,First time BST,Last time BST,Min,Max
0,HR1-181-A00-st,Actual Load,3178,2026-09-30 15:00:00.920000+01:00,2026-09-30 15:52:59.972000+01:00,-1.612329,2.459948
1,HR1-431-D00-st,Gripper Head Loaded With Pipe,40,2026-09-30 15:01:17.201000+01:00,2026-09-30 15:52:09.206000+01:00,0.000000,1.000000
2,HR1-435-D00-st,Main Arm Gripper Open,40,2026-09-30 15:00:56.732000+01:00,2026-09-30 15:52:16.221000+01:00,0.000000,1.000000
3,HR1-434-D00-st,Main Arm Gripper Close,40,2026-09-30 15:00:58.186000+01:00,2026-09-30 15:52:15.706000+01:00,0.000000,1.000000
4,HR1-352-A00-st,Lower Trolley Actual Position,1392,2026-09-30 15:00:01.420000+01:00,2026-09-30 15:52:59.675000+01:00,4.014541,6.593625


### How to inspect timing

1. Drag across a short time interval to zoom in.
2. Click **500 ms grid ON**.
3. Move the cursor left/right: the unified hover line aligns all five signals at the same timestamp.
4. Hover labels display time as `HH:MM:SS.mmm`, so the original event timing is retained rather than rounded.
5. Double-click the plot to reset the zoom.

## 6. Export the synchronized plot as interactive HTML

In [31]:
report_name = "HydraRacker_HR1_5_Signal_Synchronised_Overlay.html"

pio.write_html(
    overlay_fig,
    file=report_name,
    include_plotlyjs="cdn",
    full_html=True,
    config={"displaylogo": False, "scrollZoom": True, "responsive": True},
)

print(f"Created: {report_name}")

Created: HydraRacker_HR1_5_Signal_Synchronised_Overlay.html


## 7. Save a compact summary of the five selected signals

In [32]:
summary_name = "HydraRacker_HR1_5_Signal_Overlay_Summary.xlsx"
summary_to_save = summary.copy()
summary_to_save["First time BST"] = summary_to_save["First time BST"].dt.tz_localize(None)
summary_to_save["Last time BST"] = summary_to_save["Last time BST"].dt.tz_localize(None)
summary_to_save.to_excel(summary_name, index=False)
print(f"Created: {summary_name}")

Created: HydraRacker_HR1_5_Signal_Overlay_Summary.xlsx


## 8. Optional: export the five selected raw signal series to Excel

In [33]:
excel_name = "HydraRacker_HR1_5_Signal_Overlay_Data.xlsx"

selected_export = data.loc[data["Tag"].isin(tags)].copy()
selected_export["Time BST"] = selected_export["Time BST"].dt.tz_localize(None)
selected_export = selected_export.sort_values("Time BST")
selected_export.to_excel(excel_name, index=False)
print(f"Created: {excel_name}")

Created: HydraRacker_HR1_5_Signal_Overlay_Data.xlsx


## 9. Download the outputs

In [34]:
files.download(report_name)
files.download(summary_name)
files.download(excel_name)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>